In [ ]:
import openmc
openmc.config['cross_sections'] = '/home/elhareefmh/openmc/XS/endfb-viii.0-hdf5/cross_sections.xml'
import onix

In [ ]:
fuel = openmc.Material(temperature=973.15)
fuel.set_density('g/cm3', 4.1249)
fuel.add_nuclide('Li6', 1.4486e-05, 'ao')
fuel.add_nuclide('Li7', 0.28970514, 'ao')
fuel.add_nuclide('Th232', 0.07471028, 'ao')
fuel.add_nuclide('U233', 0.009401869, 'ao')
fuel.add_nuclide('F19', 0.626168224, 'ao')

fertile = openmc.Material(temperature=973.15)
fertile.set_density('g/cm3', 4.1249)
fertile.add_nuclide('Li6', 1.4486e-05, 'ao')
fertile.add_nuclide('Li7', 0.28970514, 'ao')
fertile.add_nuclide('Th232', 0.08411214953, 'ao')
fertile.add_nuclide('F19', 0.6261682243, 'ao')

b4c = openmc.Material(temperature=973.15)
b4c.set_density('g/cm3', 2.52)
b4c.add_nuclide('C12', 0.19778, 'ao')
b4c.add_nuclide('C13', 0.00222, 'ao')
b4c.add_nuclide('B10', 0.1584, 'ao')
b4c.add_nuclide('B11', 0.6416, 'ao')

reflector = openmc.Material(temperature=973.15)
reflector.set_density('g/cm3', 10)
reflector.add_element('Ni', 0.79432, 'ao')
reflector.add_element('W', 0.09976, 'ao')
reflector.add_element('Cr', 0.08014, 'ao')
reflector.add_element('Mo', 0.00736, 'ao')
reflector.add_element('Fe', 0.00632, 'ao')
reflector.add_element('Ti', 0.00295, 'ao')
reflector.add_element('C', 0.00294, 'ao')
reflector.add_element('Mn', 0.00257, 'ao')
reflector.add_element('Si', 0.00252, 'ao')
reflector.add_element('Al', 0.00052, 'ao')
reflector.add_element('B', 0.00033, 'ao')
reflector.add_element('P', 0.00023, 'ao')
reflector.add_element('S', 0.00003, 'ao')

gas = openmc.Material(temperature=973.15)
gas.set_density('g/cm3', 1.0)
gas.add_nuclide('He4', 1, 'ao')

materials = openmc.Materials([fuel, fertile, b4c, reflector, gas])
materials.cross_sections = '/home/elhareefmh/openmc/XS/endfb-viii.0-hdf5/cross_sections.xml'

s1 = openmc.ZCylinder(r=112.75)
s2 = openmc.ZCylinder(r=114.75)
s3 = openmc.ZCylinder(r=160.75)
s4 = openmc.ZCylinder(r=162.75)
s5 = openmc.ZCylinder(r=182.75)
s6 = openmc.ZCylinder(r=206.45)
s7 = openmc.ZCylinder(r=226.45)

h1 = openmc.ZPlane(z0=0.0)
h2 = openmc.ZPlane(z0=100.0)
h3 = openmc.ZPlane(z0=118.75)
h4 = openmc.ZPlane(z0=120.75)
h5 = openmc.ZPlane(z0=304.75)
h6 = openmc.ZPlane(z0=306.75)
h7 = openmc.ZPlane(z0=325.5)
h8 = openmc.ZPlane(z0=425.5)
h9 = openmc.ZPlane(z0=425.7483)

core_region = -s1 & +h2 & -h7
loop_region = -s6 & +h2 & -h7 & ~ ( -s5 & +h3 & -h6)
blanket_region = -s3 & +s2 & +h4 & -h5
struct_region = -s4 & +s1 & +h3 & -h6 & ~ blanket_region
protector_region = -s5 & +s4 & +h3 & - h6
reflector_region = -s7 & +h1 & -h8 & ~ (-s6 & +h2 & -h7)
reprocessing_region = -s7 & +h8 & -h9


s7.boundary_type = 'vacuum'
h1.boundary_type = 'vacuum'
h8.boundary_type = 'vacuum'

cell1 = openmc.Cell(name = 'core', region=core_region, fill=fuel)
cell2 = openmc.Cell(name = 'loop', region=loop_region, fill=fuel)
cell3 = openmc.Cell(name = 'blanket', region=blanket_region, fill=fertile)
cell4 = openmc.Cell(region=struct_region, fill=reflector)
cell5 = openmc.Cell(region=protector_region, fill=b4c)
cell6= openmc.Cell(region=reflector_region, fill=reflector)
cell7= openmc.Cell(name = 'reprocessing_plant',region=reprocessing_region, fill=fuel)

geo_universe = openmc.Universe()
geo_universe.add_cells([cell1, cell2, cell3, cell4, cell5, cell6, cell7])

# Create root Cell
root_cell = openmc.Cell(name='root cell')
root_cell.fill = geo_universe
root_cell.region = -s7 & +h1 & -h9

# Create root Universe
root_universe = openmc.Universe(universe_id=0, name='root universe')
root_universe.add_cell(root_cell)

root_universe.plot(width=(500, 500), pixels=40000, basis='xz', color_by='material')
root_universe.plot(color_by='material')

geometry = openmc.Geometry(root=root_universe)

In [ ]:
settings = openmc.Settings()
settings.run_mode='eigenvalue'
space = openmc.stats.Point((0.0, 0.0, 212.75))
source = openmc.IndependentSource(space=space)
settings.source = source
settings.batches = 120
settings.inactive = 40
settings.particles = 10000

settings.temperature['method'] = 'interpolation'
settings.temperature['tolerance'] = 100.0
settings.temperature['multipole'] = True

settings.export_to_xml()
geometry.export_to_xml()
materials.export_to_xml()
#openmc.run()

In [ ]:
import math
macrostep_vector = [1e-6, 1e-4, 1e-2, 0.5, 1, 3, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 170, 200] 
macrostep_unit = 'y'

norma_vector = [3.5e15]*len(macrostep_vector) #kW/l
norma_unit = 'flux'

periods = [macrostep_vector[i] - macrostep_vector[i-1] if i>0 else macrostep_vector[i] for i in range(len(macrostep_vector))]
microstep_vector = [math.ceil(period) for period in periods]

sequence1 = onix.Sequence(1)
sequence1.set_macrostep(macrostep_vector, macrostep_unit)
sequence1.set_norma(norma_vector, norma_unit)
sequence1.microstep_vector = microstep_vector

sequence1.flux_approximation ='iv'

msr = onix.couple.Couple_msr()

msr.set_bounding_box([-226.45, -226.45, 0.0], [226.45, 226.45, 425.5])

loop_dict = {
             'core': {
                      'upstream_cells': [('loop', 1)],
                      'elements_removal': [('H', 0.0231), ('He', 0.0231), ('N', 0.0231), ('O', 0.0231), 
                                            ('Ne', 0.0231), ('Ar', 0.0231), ('Kr', 0.0231), ('Nb', 0.0231), 
                                            ('Mo', 0.0231), ('Tc', 0.0231), ('Ru', 0.0231), ('Rh', 0.0231), 
                                            ('Pd', 0.0231), ('Ag', 0.0231), ('Sb', 0.0231), ('Te', 0.0231), 
                                            ('Xe', 0.0231), ('Rn', 0.0231)
                                            ] 
             },
             'loop': {
                     'upstream_cells': [('core',0.99999989711), ('reprocessing_plant',1.0289e-07)]
             },
             'reprocessing_plant': {
                     'upstream_cells': [('core',1.0289e-07)],
                     'elements_removal': [('Zn', 1.1574e-04), ('Ga', 1.1574e-04), ('Ge', 1.1574e-04), ('As', 1.1574e-04), 
                                            ('Se', 1.1574e-04), ('Br', 1.1574e-04), ('Rb', 1.1574e-04), ('Sr', 1.1574e-04), 
                                            ('Y', 1.1574e-04), ('Zr', 1.1574e-04), ('Cd', 1.1574e-04), ('In', 1.1574e-04), 
                                            ('Sn', 1.1574e-04), ('I', 1.1574e-04), ('Cs', 1.1574e-04), ('Ba', 1.1574e-04), 
                                            ('La', 1.1574e-04), ('Ce', 1.1574e-04), ('Pr', 1.1574e-04), ('Nd', 1.1574e-04), 
                                            ('Pm', 1.1574e-04), ('Sm', 1.1574e-04), ('Eu', 1.1574e-04), ('Gd', 1.1574e-04), 
                                            ('Tb', 1.1574e-04), ('Dy', 1.1574e-04), ('Ho', 1.1574e-04), ('Er', 1.1574e-04), 
                                            ('Tm', 1.1574e-04), ('Yb', 1.1574e-04)
                                            ],
             },
            'blanket':{
                    'elements_removal': [('H', 0.0231), ('He', 0.0231), ('N', 0.0231), ('O', 0.0231), 
                                            ('Ne', 0.0231), ('Ar', 0.0231), ('Kr', 0.0231), ('Nb', 0.0231), 
                                            ('Mo', 0.0231), ('Tc', 0.0231), ('Ru', 0.0231), ('Rh', 0.0231), 
                                            ('Pd', 0.0231), ('Ag', 0.0231), ('Sb', 0.0231), ('Te', 0.0231), 
                                            ('Xe', 0.0231), ('Rn', 0.0231),   #Nobel gases 
                                            ('Zn', 6.3197e-10), ('Ga', 6.3197e-10), ('Ge', 6.3197e-10), ('As', 6.3197e-10), 
                                            ('Se', 6.3197e-10), ('Br', 6.3197e-10), ('Rb', 6.3197e-10), ('Sr', 6.3197e-10), 
                                            ('Y', 6.3197e-10), ('Zr', 6.3197e-10), ('Cd', 6.3197e-10), ('In', 6.3197e-10), 
                                            ('Sn', 6.3197e-10), ('I', 6.3197e-10), ('Cs', 6.3197e-10), ('Ba', 6.3197e-10), 
                                            ('La', 6.3197e-10), ('Ce', 6.3197e-10), ('Pr', 6.3197e-10), ('Nd', 6.3197e-10), 
                                            ('Pm', 6.3197e-10), ('Sm', 6.3197e-10), ('Eu', 6.3197e-10), ('Gd', 6.3197e-10), 
                                            ('Tb', 6.3197e-10), ('Dy', 6.3197e-10), ('Ho', 6.3197e-10), ('Er', 6.3197e-10), 
                                            ('Tm', 6.3197e-10), ('Yb', 6.3197e-10),
                                            ('U', 6.3197e-08), ('Np', 6.3197e-08), 
                                            ('Pu', 6.3197e-08), ('Am', 6.3197e-08), ('Cm', 6.3197e-08), ('Bk', 6.3197e-08), 
                                            ('Cf', 6.3197e-08), ('Es', 6.3197e-08), ('Fm', 6.3197e-08), ('Md', 6.3197e-08), 
                                            ('No', 6.3197e-08), ('Lr', 6.3197e-08)
                                            ],
                
                    'recycling': ('core', [('U', 6.3197e-08), ('Np', 6.3197e-08), 
                                            ('Pu', 6.3197e-08), ('Am', 6.3197e-08), ('Cm', 6.3197e-08), ('Bk', 6.3197e-08), 
                                            ('Cf', 6.3197e-08), ('Es', 6.3197e-08), ('Fm', 6.3197e-08), ('Md', 6.3197e-08), 
                                            ('No', 6.3197e-08), ('Lr', 6.3197e-08)
                                            ])
                        }
}
msr.vol_flow_rate = 4.5e6

msr.select_bucells(loop_dict = loop_dict)

msr.import_openmc(root_cell)

msr.set_sequence(sequence1)

vol_dict = {'core': 9005949.76, 'loop': 8.9715e+06, 'reprocessing_plant':40E3, 'blanket':7.3257e+06, 'total volume':6.8548e+07}
msr.set_vol(vol_dict)




msr.burn()